# Sentiment Analysis — ML Pipeline (cached data)

Input: `Data/combined_sentiment_cleaned.tsv` (already cleaned)
Output: `Models/countVectorizer.pkl`, `Models/model_xgb.pkl`, `Predictions-new.csv`

### Importing required libraries

In [2]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_score,
    recall_score,
    f1_score,
)

CLEANED_PATH    = "Data/combined_sentiment.tsv"
MODELS_DIR      = "Models"
VECTORIZER_PATH = f"{MODELS_DIR}/countVectorizer.pkl"
MODEL_PATH      = f"{MODELS_DIR}/model_xgb.pkl"
PREDICTIONS_CSV = "Predictions-new.csv"

os.makedirs(MODELS_DIR, exist_ok=True)
print("Paths ready.")

Paths ready.


In [3]:
df = pd.read_csv(CLEANED_PATH, sep="\t")

print("Shape:", df.shape)
print("Columns:", list(df.columns))
print(df["label"].value_counts())
print()
print(df.head(3))

Shape: (4252349, 2)
Columns: ['review', 'label']
label
Positive    2130069
Negative    2122280
Name: count, dtype: int64

                                              review     label
0  well, we went there at 10pm on a friday and th...  Positive
1  I'm cleaning up my draft reviews and was here ...  Negative
2  I wish I could imagine what it would be like n...  Positive


In [4]:
MAX_PER_CLASS = 400_000   # set to None to use the full dataset

if MAX_PER_CLASS is not None:
    pos = df[df["label"] == "Positive"].sample(MAX_PER_CLASS, random_state=42)
    neg = df[df["label"] == "Negative"].sample(MAX_PER_CLASS, random_state=42)
    df = pd.concat([pos, neg]).sample(frac=1, random_state=42).reset_index(drop=True)

print("Working shape:", df.shape)
print(df["label"].value_counts())

Working shape: (800000, 2)
label
Positive    400000
Negative    400000
Name: count, dtype: int64


In [ ]:
tfidf = TfidfVectorizer(
    max_features=30_000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.9,
    sublinear_tf=True,
    dtype=np.float32,
)

print("Vectorizing...")
X = tfidf.fit_transform(df["review"])           # sparse — do NOT call .toarray()
y = (df["label"] == "Positive").astype(np.int8).values

print("X shape:", X.shape, "| sparse:", hasattr(X, "toarray"))
print("y shape:", y.shape, "| dtype:", y.dtype)

pickle.dump(tfidf, open(VECTORIZER_PATH, "wb"))
print("Saved vectorizer:", VECTORIZER_PATH)

Vectorizing...


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)

print("Train:", X_train.shape, y_train.shape)
print("Test :", X_test.shape, y_test.shape)
print("Train balance:", np.bincount(y_train) / len(y_train))
print("Test  balance:", np.bincount(y_test)  / len(y_test))

product not good not happy
product really good happy


In [ ]:
base_svc = LinearSVC(class_weight="balanced", C=1.0, random_state=42)
model_svc = CalibratedClassifierCV(base_svc, cv=3)

print("Training LinearSVC...")
model_svc.fit(X_train, y_train)
print("Done.")

In [ ]:
def evaluate_model(name, model, X_te, y_te,
                   labels=(0, 1),
                   target_names=("Negative (0)", "Positive (1)")):
    y_pred = model.predict(X_te)
    acc = accuracy_score(y_te, y_pred)

    print(f"===== {name} =====")
    print(f"Test Accuracy : {acc:.4f}\n")
    print(classification_report(
        y_te, y_pred,
        labels=list(labels),
        target_names=list(target_names),
        zero_division=0,
    ))

    cm = confusion_matrix(y_te, y_pred, labels=list(labels))
    ConfusionMatrixDisplay(cm, display_labels=list(target_names)).plot()
    plt.title(f"Confusion Matrix - {name}")
    plt.show()

    metrics = {
        "test_accuracy":      acc,
        "negative_precision": precision_score(y_te, y_pred, pos_label=0, zero_division=0),
        "negative_recall":    recall_score   (y_te, y_pred, pos_label=0, zero_division=0),
        "negative_f1":        f1_score       (y_te, y_pred, pos_label=0, zero_division=0),
        "positive_precision": precision_score(y_te, y_pred, pos_label=1, zero_division=0),
        "positive_recall":    recall_score   (y_te, y_pred, pos_label=1, zero_division=0),
        "positive_f1":        f1_score       (y_te, y_pred, pos_label=1, zero_division=0),
    }
    metrics["macro_f1"] = (metrics["negative_f1"] + metrics["positive_f1"]) / 2
    return metrics, y_pred


metrics, y_preds = evaluate_model("LinearSVC", model_svc, X_test, y_test)
print(metrics)

X shape: (3149, 5000)
y shape: (3149,)


In [ ]:
pickle.dump(model_svc, open(MODEL_PATH, "wb"))
print("Saved model:", MODEL_PATH)

Train: (2519, 5000) (2519,)
Test : (630, 5000) (630,)
Train balance: {1: 0.9186, 0: 0.0814}
Test balance : {1: 0.919, 0: 0.081}


In [ ]:
import re
import nltk
from nltk.corpus import stopwords
nltk.download("stopwords", quiet=True)

CONTRACTION_MAP = {
    "ain't": "is not", "aren't": "are not", "can't": "cannot", "can't've": "cannot have",
    "could've": "could have", "couldn't": "could not", "didn't": "did not",
    "doesn't": "does not", "don't": "do not", "hadn't": "had not", "hasn't": "has not",
    "haven't": "have not", "he'd": "he would", "he'll": "he will", "he's": "he is",
    "how'd": "how did", "how'll": "how will", "how's": "how is", "i'd": "i would",
    "i'll": "i will", "i'm": "i am", "i've": "i have", "isn't": "is not",
    "it'd": "it would", "it'll": "it will", "it's": "it is", "let's": "let us",
    "mightn't": "might not", "might've": "might have", "mustn't": "must not",
    "must've": "must have", "needn't": "need not", "shan't": "shall not",
    "she'd": "she would", "she'll": "she will", "she's": "she is",
    "should've": "should have", "shouldn't": "should not", "that'd": "that would",
    "that's": "that is", "there'd": "there would", "there's": "there is",
    "they'd": "they would", "they'll": "they will", "they're": "they are",
    "they've": "they have", "wasn't": "was not", "we'd": "we would",
    "we'll": "we will", "we're": "we are", "we've": "we have", "weren't": "were not",
    "what'll": "what will", "what're": "what are", "what's": "what is",
    "what've": "what have", "where'd": "where did", "where's": "where is",
    "who'll": "who will", "who's": "who is", "won't": "will not", "would've": "would have",
    "wouldn't": "would not", "you'd": "you would", "you'll": "you will",
    "you're": "you are", "you've": "you have",
}
CONTRACTION_PATTERN = re.compile(
    r"\b(" + "|".join(re.escape(k) for k in CONTRACTION_MAP) + r")\b",
    flags=re.IGNORECASE,
)

NEGATION_WORDS = {
    "no", "nor", "not", "never", "none", "nothing", "nowhere", "neither",
    "without", "cannot", "can", "against",
}
CUSTOM_STOPWORDS = set(stopwords.words("english")) - NEGATION_WORDS


def clean_review(text: str) -> str:
    text = str(text).lower()
    text = CONTRACTION_PATTERN.sub(
        lambda m: CONTRACTION_MAP.get(m.group(0).lower(), m.group(0)), text
    )
    text = re.sub("[^a-zA-Z]", " ", text)
    words = [w for w in text.split() if w not in CUSTOM_STOPWORDS]
    return " ".join(words)


def predict_sentiment(sentence: str) -> str:
    cleaned = clean_review(sentence)
    vec = tfidf.transform([cleaned])
    pred = model_svc.predict(vec)[0]
    return "Positive" if pred == 1 else "Negative"

In [ ]:
test_sentences = [
    "I am really happy with the quality of this product.",
    "The product did not meet my expectations.",
    "It is not worth the money.",
    "Amazing experience, I am completely satisfied.",
    "I could not get any useful help from support.",
    "Excellent build quality and great design.",
]

for s in test_sentences:
    print(f"{s} -> {predict_sentiment(s)}")

Before SMOTE: {1: 2314, 0: 205}
After  SMOTE: {1: 2314, 0: 2314}


In [ ]:
tfidf = pickle.load(open(VECTORIZER_PATH, "rb"))
model_svc = pickle.load(open(MODEL_PATH, "rb"))
print("Reloaded vectorizer and model.")

In [ ]:
import pickle
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score

# 1. Load the cleaned TSV and check the label column
df = pd.read_csv("Data/combined_sentiment.tsv", sep="\t")
print("Label dtype:", df["label"].dtype)
print("Label values:", df["label"].unique()[:10])
print()

# 2. Check the vectorizer + model
tfidf = pickle.load(open("Models/countVectorizer.pkl", "rb"))
model = pickle.load(open("Models/model_xgb.pkl", "rb"))

# 3. What classes does the model know?
print("Model classes_:", model.classes_)

# 4. Encode labels the same way as during training
if df["label"].dtype == object:
    y_true = (df["label"] == "Positive").astype(int).values
else:
    # labels are integers, so decide the convention NOW
    y_true = df["label"].astype(int).values
    print("Label integers found. Assuming 1 = Positive, 0 = Negative.")
    print("If your file was built the other way round, the mapping is reversed.")

# 5. Predict on a subset and check accuracy
X = tfidf.transform(df["review"].head(2000))
y_pred = model.predict(X)
print("Quick accuracy on first 2000 rows:", accuracy_score(y_true[:2000], y_pred))

# 6. Sanity check — what does the model say about obvious sentences?
for s in ["I am happy", "I am not happy", "terrible product", "excellent product"]:
    cleaned = s.lower()
    p = model.predict(tfidf.transform([cleaned]))[0]
    print(f"{s!r:30} -> pred class {p}")